In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import BoxLeastSquares
from astropy.timeseries import LombScargle

from wotan import flatten

# ============================================================
# USER PARAMETERS
# ============================================================

WINDOW_LENGTH = 2.0
WIDTH_FACTOR  = 3.0

MIN_PERIOD_BLS = 0.3
MAX_PERIOD_BLS = 15.0

MIN_PERIOD_LS = 0.2
MAX_PERIOD_LS = 15.0

# ============================================================
# LOAD LIGHT CURVE
# ============================================================
#llamar aqui con el nombre de la columna correspondiente en el archivo a utilizar
def load_lightcurve(
        filename,
        flux_column="Flux_CBV_SS" 
):

    df = pd.read_csv(
        filename,
        sep=r"\s+"
    )

    time = df["#Time"].values
    flux = df[flux_column].values

    mask = (
        np.isfinite(time)
        &
        np.isfinite(flux)
    )

    return (
        time[mask],
        flux[mask]
    )

# ============================================================
# BLS
# ============================================================

def run_bls(
        time,
        flux
):

    periods = np.linspace(
        MIN_PERIOD_BLS,
        MAX_PERIOD_BLS,
        10000
    )

    durations = np.linspace(
        0.01,
        0.08,
        20
    )

    bls = BoxLeastSquares(
        time,
        flux
    )

    result = bls.power(
        periods,
        durations
    )

    idx = np.argmax(
        result.power
    )

    return {

        "P_bls":
            result.period[idx],

        "t0":
            result.transit_time[idx],

        "duration":
            result.duration[idx],

        "period_grid":
            result.period,

        "power_grid":
            result.power
    }

# ============================================================
# P OR 2P ?
# ============================================================

def determine_orbital_period(
        time,
        flux,
        t0,
        P_bls
):

    P1 = P_bls
    P2 = 2 * P_bls

    phase = (
        (time - t0)
        / P2
    ) % 1

    primary = np.nanmedian(
        flux[
            np.abs(
                phase - 0.0
            ) < 0.05
        ]
    )

    secondary = np.nanmedian(
        flux[
            np.abs(
                phase - 0.5
            ) < 0.05
        ]
    )

    if abs(
        primary-secondary
    ) > 0.005:

        return P2

    return P1

# ============================================================
# ECLIPSE MASK
# ============================================================

def build_mask(
        time,
        period,
        t0,
        duration,
        width_factor=3.0
):

    phase = (
        (time - t0)
        / period
    ) % 1

    width = (
        width_factor
        * duration
        / period
    )

    primary = (
        (phase < width)
        |
        (phase > 1-width)
    )

    secondary = (
        np.abs(
            phase - 0.5
        ) < width
    )

    return (
        primary
        |
        secondary
    )

# ============================================================
# LOMB SCARGLE
# ============================================================

def run_ls(
        time,
        flux
):

    freq = np.linspace(
        1/MAX_PERIOD_LS,
        1/MIN_PERIOD_LS,
        10000
    )

    ls = LombScargle(
        time,
        flux
    )

    power = ls.power(
        freq
    )

    idx = np.argmax(
        power
    )

    return {

        "P_rot":
            1/freq[idx],

        "frequency":
            freq,

        "power":
            power
    }

# ============================================================
# MAIN PIPELINE
# ============================================================

def analyze_system(
        filename,
        flux_column="Flux_CBV_SS",
        window_length=WINDOW_LENGTH,
        width_factor=WIDTH_FACTOR
):

    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    time, flux = load_lightcurve(
        filename,
        flux_column
    )

    # --------------------------------------------------------
    # DETREND
    # --------------------------------------------------------

    orbital_flux, rotational_trend = flatten(
        time,
        flux,
        method="biweight",
        window_length=window_length,
        break_tolerance=0.5,
        return_trend=True
    )

    # --------------------------------------------------------
    # BLS
    # --------------------------------------------------------

    orbital = run_bls(
        time,
        orbital_flux
    )

    P_orb = determine_orbital_period(
        time,
        orbital_flux,
        orbital["t0"],
        orbital["P_bls"]
    )

    # --------------------------------------------------------
    # MASK
    # --------------------------------------------------------

    mask = build_mask(
        time,
        P_orb,
        orbital["t0"],
        orbital["duration"],
        width_factor
    )

    # --------------------------------------------------------
    # ROTATION
    # --------------------------------------------------------

    time_rot = time[~mask]

    flux_rot = flux[~mask]

    rotation = run_ls(
        time_rot,
        flux_rot
    )

    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    return {

        "time": time,
        "flux": flux,

        "orbital_flux":
            orbital_flux,

        "rotational_flux":
            rotational_trend,

        "time_rot":
            time_rot,

        "flux_rot":
            flux_rot,

        "mask":
            mask,

        "P_bls":
            orbital["P_bls"],

        "P_orb":
            P_orb,

        "P_rot":
            rotation["P_rot"],

        "t0":
            orbital["t0"],

        "duration":
            orbital["duration"],

        "orbital":
            orbital,

        "rotation":
            rotation,

        "window_length":
            window_length
    }



In [ ]:
# ============================================================
# PAPER FIGURE
# ============================================================

def figure_paper(results):

    import numpy as np
    import matplotlib.pyplot as plt

    # --------------------------------------------------------
    # DATA
    # --------------------------------------------------------

    time = results["time"]
    flux = results["flux"]

    orbital_flux = results["orbital_flux"]
    rotational_flux = results["rotational_flux"]

    time_rot = results["time_rot"]
    flux_rot = results["flux_rot"]

    P_orb = results["P_orb"]
    P_rot = results["P_rot"]

    t0 = results["t0"]

    # --------------------------------------------------------
    # FASES
    # --------------------------------------------------------

    phase_orb = (
        (time - t0)
        / P_orb
    ) % 1

    # La rotación debe calcularse usando la tendencia Wotan
    rot_signal = rotational_flux / np.nanmedian(
        rotational_flux
    )

    phase_rot = (
        (time - np.nanmin(time))
        / P_rot
    ) % 1

    idx_orb = np.argsort(phase_orb)
    idx_rot = np.argsort(phase_rot)

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        2,
        2,
        figsize=(14,10)
    )

    # ========================================================
    # (a) rotacional
    # ========================================================

    ax[0,0].plot(
        time,
        flux,
        ".",
        ms=1,
        alpha=0.25,
        label="Original"
    )

    ax[0,0].plot(
        time,
        rotational_flux,
        lw=2,
        label="Wotan trend"
    )

    ax[0,0].set_title(
        "(a) Detrending"
    )

    ax[0,0].set_xlabel("Time (days)")
    ax[0,0].set_ylabel("Flux")

    ax[0,0].legend()

       # ========================================================
    # (b) FASE ROTACIONAL
    # ========================================================

    ax[0,1].plot(
        phase_rot[idx_rot],
        rot_signal[idx_rot],
        ".",
        ms=2,
        alpha=0.4
    )

    ax[0,1].set_title(
        f"(b) Fase rotacional\nP_rot = {P_rot:.5f} d"
    )

    ax[0,1].set_xlabel("Phase")
    ax[0,1].set_ylabel("Normalized flux")

    # ========================================================
    # (c) ORBITAL
    # ========================================================

    ax[1,0].plot(
        time,
        orbital_flux,
        ".",
        ms=2,
        alpha=0.5
    )

    ax[1,0].set_title(
        "(c) Curva aplanada"
    )

    ax[1,0].set_xlabel("Time (days)")
    ax[1,0].set_ylabel("Normalized flux")

    # ========================================================
    # (d) fase
    # ========================================================

    ax[1,1].plot(
        phase_orb[idx_orb],
        orbital_flux[idx_orb],
        ".",
        ms=2,
        alpha=0.5
    )

    ax[1,1].set_title(
        f"(d) Fase\nP_orb = {P_orb:.5f} d"
    )

    ax[1,1].set_xlabel("Phase")
    ax[1,1].set_ylabel("Normalized flux")

    # ========================================================
    #  Titulo
    # ========================================================

    fig.suptitle(
        f"P_orb = {P_orb:.5f} d   |   "
        f"P_rot = {P_rot:.5f} d   |   "
        f"Window = {results['window_length']:.2f} d",
        fontsize=14
    )

    plt.tight_layout()

    plt.show()

# ============================================================
# EXAMPLE
# ============================================================

file = "/home/luciam/Documents/starry/Orion_yceb/dat_orion_sergio/sample_3/831359(1)/Light_Curve_Files/tic 11300044_S_98_LC.txt"

results = analyze_system(
    file,
    window_length=.9
)

print()
print(f"P_BLS = {results['P_bls']:.6f} d")
print(f"P_orb = {results['P_orb']:.6f} d")
print(f"P_rot = {results['P_rot']:.6f} d")

In [ ]:
figure_paper(results)

# Plots mas bonitos

In [ ]:
colors = {

    "rot": "#1f77b4",      # azul
    "orb": "#7b3294",      # púrpura
    "trend": "#ff7f0e",    # naranja
    "bin": "#111111"       # negro suave
}

In [ ]:
# ============================================================
# PUBLICATION FIGURE
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ============================================================
# DATA
# ============================================================

time = results["time"]
flux = results["flux"]

orbital_flux = results["orbital_flux"]
rotational_flux = results["rotational_flux"]

P_orb = results["P_orb"]
P_rot = results["P_rot"]
t0 = results["t0"]

# ============================================================
# WOTAN TREND (DISPLAY ONLY)
# ============================================================

window = 501

trend = np.convolve(
    flux,
    np.ones(window) / window,
    mode="same"
)

# ============================================================
# ORBITAL PHASE
# ============================================================

phase_orb = (
    ((time - t0) % P_orb)
    / P_orb
)

idx_orb = np.argsort(
    phase_orb
)

# ============================================================
# BINNED ORBITAL CURVE
# ============================================================

nbins_orb = 100

bins_orb = np.linspace(
    0,
    1,
    nbins_orb + 1
)

digit_orb = np.digitize(
    phase_orb,
    bins_orb
)

phase_bin_orb = []
flux_bin_orb = []

for k in range(1, len(bins_orb)):

    mask = digit_orb == k

    if np.sum(mask) > 10:

        phase_bin_orb.append(
            np.nanmedian(
                phase_orb[mask]
            )
        )

        flux_bin_orb.append(
            np.nanmedian(
                orbital_flux[mask]
            )
        )

phase_bin_orb = np.array(
    phase_bin_orb
)

flux_bin_orb = np.array(
    flux_bin_orb
)

# ============================================================
# ROTATIONAL PHASE
# ============================================================

phase_rot = (
    (time - time[0])
    / P_rot
) % 1.0

idx_rot = np.argsort(
    phase_rot
)

# ============================================================
# BINNED ROTATIONAL CURVE
# ============================================================

nbins_rot = 50

bins_rot = np.linspace(
    0,
    1,
    nbins_rot + 1
)

digit_rot = np.digitize(
    phase_rot,
    bins_rot
)

phase_bin_rot = []
flux_bin_rot = []

for k in range(1, len(bins_rot)):

    mask = digit_rot == k

    if np.sum(mask) > 20:

        phase_bin_rot.append(
            np.nanmedian(
                phase_rot[mask]
            )
        )

        flux_bin_rot.append(
            np.nanmedian(
                rotational_flux[mask]
            )
        )

phase_bin_rot = np.array(
    phase_bin_rot
)

flux_bin_rot = np.array(
    flux_bin_rot
)

# ============================================================
# FIGURE LAYOUT
# ============================================================

fig = plt.figure(
    figsize=(12,12)
)

gs = GridSpec(
    4,
    2,
    height_ratios=[1,1,1,0.45],
    hspace=0.40,
    wspace=0.20
)

# ------------------------------------------------------------
# AXES
# ------------------------------------------------------------

ax_a = fig.add_subplot(gs[0,0])
ax_b = fig.add_subplot(gs[0,1])

ax_c = fig.add_subplot(gs[1,0])
ax_d = fig.add_subplot(gs[1,1])

ax_e = fig.add_subplot(gs[2,:])

ax_table = fig.add_subplot(gs[3,:])

# ============================================================
# (a) ORIGINAL LIGHT CURVE
# ============================================================

ax_a.plot(
    time,
    flux,
    ".",
    ms=1,
    alpha=0.12,
    color="dodgerblue",
    rasterized=True
)

ax_a.plot(
    time,
    trend,
    color="deeppink",
    lw=2
)

ax_a.set_title(
    "(a) Original light curve + Wotan trend"
)

ax_a.set_xlabel(
    "Time [d]"
)

ax_a.set_ylabel(
    "Flux"
)

# ============================================================
# (b) ROTATIONAL COMPONENT
# ============================================================

ax_b.plot(
    time,
    rotational_flux,
    ".",
    ms=1,
    alpha=0.35,
    color="steelblue",
    rasterized=True
)

ax_b.set_title(
    f"(b) Rotational component ($P_{{rot}}={P_rot:.5f}$ d)"
)

ax_b.set_xlabel(
    "Time [d]"
)

ax_b.set_ylabel(
    "Flux"
)

# ============================================================
# (c) FLATTENED ORBITAL COMPONENT
# ============================================================

ax_c.plot(
    time,
    orbital_flux,
    ".",
    ms=1,
    alpha=0.15,
    color="purple",
    rasterized=True
)

ax_c.set_title(
    "(c) Flattened orbital component"
)

ax_c.set_xlabel(
    "Time [d]"
)

ax_c.set_ylabel(
    "Normalized flux"
)

# ============================================================
# (d) ORBITAL PHASE
# ============================================================

ax_d.plot(
    phase_orb[idx_orb],
    orbital_flux[idx_orb],
    ".",
    ms=2,
    alpha=0.12,
    color="purple",
    rasterized=True
)

ax_d.plot(
    phase_bin_orb,
    flux_bin_orb,
    color="black",
    lw=2
)

ax_d.set_xlim(
    0,
    1
)

ax_d.set_title(
    f"(d) Orbital phase ($P_{{orb}}={P_orb:.5f}$ d)"
)

ax_d.set_xlabel(
    "Phase"
)

ax_d.set_ylabel(
    "Normalized flux"
)

# ============================================================
# (e) ROTATIONAL PHASE
# ============================================================

ax_e.plot(
    phase_rot[idx_rot],
    rotational_flux[idx_rot],
    ".",
    ms=1,
    alpha=0.12,
    color="steelblue",
    rasterized=True
)

ax_e.plot(
    phase_bin_rot,
    flux_bin_rot,
    color="black",
    lw=2
)

ax_e.set_xlim(
    0,
    1
)

ax_e.set_title(
    f"(e) Rotational phase ($P_{{rot}}={P_rot:.5f}$ d)"
)

ax_e.set_xlabel(
    "Phase"
)

ax_e.set_ylabel(
    "Flux"
)

# ============================================================
# TABLE
# ============================================================

ax_table.axis("off")

table_data = [
    ["P_orb (d)", f"{P_orb:.6f}"],
    ["P_rot (d)", f"{P_rot:.6f}"],
    ["T0", f"{t0:.6f}"]
]

table = ax_table.table(
    cellText=table_data,
    colLabels=["Parameter", "Value"],
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "publication_figure.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# EXAMPLE
# ============================================================

file = "/home/luciam/Documents/starry/Orion_yceb/dat_orion_sergio/sample_3/831359(1)/Light_Curve_Files/tic 50897998_S_32_LC.txt"

results = analyze_system(
    file,
    window_length=.55
)

print()
print(f"P_BLS = {results['P_bls']:.6f} d")
print(f"P_orb = {results['P_orb']:.6f} d")
print(f"P_rot = {results['P_rot']:.6f} d")

In [ ]:
figure_paper(results)

In [ ]:
time = results["time"]
flux = results["flux"]

orbital_flux = results["orbital_flux"]
rotational_flux = results["rotational_flux"]

P_orb = results["P_orb"]
P_rot = results["P_rot"]
t0 = results["t0"]

# ============================================================
# WOTAN TREND (DISPLAY ONLY)
# ============================================================

window = 501

trend = np.convolve(
    flux,
    np.ones(window) / window,
    mode="same"
)

# ============================================================
# ORBITAL PHASE
# ============================================================

phase_orb = (
    ((time - t0) % P_orb)
    / P_orb
)

idx_orb = np.argsort(
    phase_orb
)

# ============================================================
# BINNED ORBITAL CURVE
# ============================================================

nbins_orb = 100

bins_orb = np.linspace(
    0,
    1,
    nbins_orb + 1
)

digit_orb = np.digitize(
    phase_orb,
    bins_orb
)

phase_bin_orb = []
flux_bin_orb = []

for k in range(1, len(bins_orb)):

    mask = digit_orb == k

    if np.sum(mask) > 10:

        phase_bin_orb.append(
            np.nanmedian(
                phase_orb[mask]
            )
        )

        flux_bin_orb.append(
            np.nanmedian(
                orbital_flux[mask]
            )
        )

phase_bin_orb = np.array(
    phase_bin_orb
)

flux_bin_orb = np.array(
    flux_bin_orb
)

# ============================================================
# ROTATIONAL PHASE
# ============================================================

phase_rot = (
    (time - time[0])
    / P_rot
) % 1.0

idx_rot = np.argsort(
    phase_rot
)

# ============================================================
# BINNED ROTATIONAL CURVE
# ============================================================

nbins_rot = 50

bins_rot = np.linspace(
    0,
    1,
    nbins_rot + 1
)

digit_rot = np.digitize(
    phase_rot,
    bins_rot
)

phase_bin_rot = []
flux_bin_rot = []

for k in range(1, len(bins_rot)):

    mask = digit_rot == k

    if np.sum(mask) > 20:

        phase_bin_rot.append(
            np.nanmedian(
                phase_rot[mask]
            )
        )

        flux_bin_rot.append(
            np.nanmedian(
                rotational_flux[mask]
            )
        )

phase_bin_rot = np.array(
    phase_bin_rot
)

flux_bin_rot = np.array(
    flux_bin_rot
)

# ============================================================
# FIGURE LAYOUT
# ============================================================

fig = plt.figure(
    figsize=(12,12)
)

gs = GridSpec(
    4,
    2,
    height_ratios=[1,1,1,0.45],
    hspace=0.40,
    wspace=0.20
)

# ------------------------------------------------------------
# AXES
# ------------------------------------------------------------

ax_a = fig.add_subplot(gs[0,0])
ax_b = fig.add_subplot(gs[0,1])

ax_c = fig.add_subplot(gs[1,0])
ax_d = fig.add_subplot(gs[1,1])

ax_e = fig.add_subplot(gs[2,:])

ax_table = fig.add_subplot(gs[3,:])

# ============================================================
# (a) ORIGINAL LIGHT CURVE
# ============================================================

ax_a.plot(
    time,
    flux,
    ".",
    ms=1,
    alpha=0.12,
    color="dodgerblue",
    rasterized=True
)

ax_a.plot(
    time,
    trend,
    color="deeppink",
    lw=2
)

ax_a.set_title(
    "(a) Original light curve + Wotan trend"
)

ax_a.set_xlabel(
    "Time [d]"
)

ax_a.set_ylabel(
    "Flux"
)

# ============================================================
# (b) ROTATIONAL COMPONENT
# ============================================================

ax_b.plot(
    time,
    rotational_flux,
    ".",
    ms=1,
    alpha=0.35,
    color="steelblue",
    rasterized=True
)

ax_b.set_title(
    f"(b) Rotational component ($P_{{rot}}={P_rot:.5f}$ d)"
)

ax_b.set_xlabel(
    "Time [d]"
)

ax_b.set_ylabel(
    "Flux"
)

# ============================================================
# (c) FLATTENED ORBITAL COMPONENT
# ============================================================

ax_c.plot(
    time,
    orbital_flux,
    ".",
    ms=1,
    alpha=0.15,
    color="purple",
    rasterized=True
)

ax_c.set_title(
    "(c) Flattened orbital component"
)

ax_c.set_xlabel(
    "Time [d]"
)

ax_c.set_ylabel(
    "Normalized flux"
)

# ============================================================
# (d) ORBITAL PHASE
# ============================================================

ax_d.plot(
    phase_orb[idx_orb],
    orbital_flux[idx_orb],
    ".",
    ms=2,
    alpha=0.12,
    color="purple",
    rasterized=True
)

ax_d.plot(
    phase_bin_orb,
    flux_bin_orb,
    color="black",
    lw=2
)

ax_d.set_xlim(
    0,
    1
)

ax_d.set_title(
    f"(d) Orbital phase ($P_{{orb}}={P_orb:.5f}$ d)"
)

ax_d.set_xlabel(
    "Phase"
)

ax_d.set_ylabel(
    "Normalized flux"
)

# ============================================================
# (e) ROTATIONAL PHASE
# ============================================================

ax_e.plot(
    phase_rot[idx_rot],
    rotational_flux[idx_rot],
    ".",
    ms=1,
    alpha=0.12,
    color="steelblue",
    rasterized=True
)

ax_e.plot(
    phase_bin_rot,
    flux_bin_rot,
    color="black",
    lw=2
)

ax_e.set_xlim(
    0,
    1
)

ax_e.set_title(
    f"(e) Rotational phase ($P_{{rot}}={P_rot:.5f}$ d)"
)

ax_e.set_xlabel(
    "Phase"
)

ax_e.set_ylabel(
    "Flux"
)

# ============================================================
# TABLE
# ============================================================

ax_table.axis("off")

table_data = [
    ["P_orb (d)", f"{P_orb:.6f}"],
    ["P_rot (d)", f"{P_rot:.6f}"],
    ["T0", f"{t0:.6f}"]
]

table = ax_table.table(
    cellText=table_data,
    colLabels=["Parameter", "Value"],
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "publication_figure.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()